[Lab README](README.md)

# Lab 2.1: Vector retrieval, with and without the graph

This is the comparison the workshop is built on, run twice over the same index
with the same embedding.

`VectorRetriever` is standard RAG. It embeds the question, finds the nearest
chunks, and returns their text. Nothing else. Every vector database does this,
and for a paraphrased question about a fact stated in one chunk, it is enough.

`VectorCypherRetriever` starts identically, with the same vector search over the
same index, and then follows the matched chunk into the graph. The retrieved
text is the same. What comes back with it is not.

Run both. The difference is the point of the lab, and it is worth seeing rather
than being told.

In [ ]:
# At an AWS event the dependencies are already installed. Self-paced, run
# `uv venv && uv pip install -r requirements.txt` inside 02-retrieval first,
# then run this cell to confirm the kernel can see what the notebook imports.
import importlib.util
import sys

REQUIRED = ("boto3", "dotenv", "neo4j", "neo4j_graphrag", "workshop")
absent = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
if absent:
    raise ModuleNotFoundError(
        f"Not importable: {', '.join(absent)}. Install this lab's "
        "requirements.txt, then restart the kernel."
    )

print(f"Python {sys.version_info.major}.{sys.version_info.minor}")
print(f"Imports resolve: {', '.join(REQUIRED)}")

## Connect and verify the graph

Retrieval notebooks do not create schema artifacts. This cell requires both Lab 1
indexes to be online with the expected label, property, dimensions, and
similarity function, then checks the graph facts the questions below depend on.

In [ ]:
import os

import boto3
from dotenv import load_dotenv
from IPython.display import HTML, display
from neo4j import GraphDatabase

load_dotenv()

NEO4J_VARS = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
missing = [name for name in NEO4J_VARS if not os.environ.get(name)]
has_aws = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = not missing and has_aws

if missing:
    print(f"Neo4j is not configured: set {', '.join(missing)} in the repo-root .env")
if not has_aws:
    print("No AWS credentials found, so the Bedrock calls below cannot run")

if RETRIEVAL_READY:
    # Imported here because `workshop.graph_connection` raises at import when
    # NEO4J_PASSWORD is unset, which would fail this cell instead of skipping it.
    from workshop.graph_connection import NEO4J_URI, neo4j_auth
    from workshop.retrieval_contract import (
        CHUNK_FULLTEXT_INDEX,
        CHUNK_VECTOR_INDEX,
        EMBEDDING_DIMENSIONS,
        EMBEDDING_MODEL_ID,
    )
    from workshop.retrieval_setup import fixture_problems, verify_retrieval_indexes

    driver = GraphDatabase.driver(NEO4J_URI, auth=neo4j_auth())
    driver.verify_connectivity()

    try:
        verify_retrieval_indexes(driver)
        problems = fixture_problems(driver)
        if problems:
            raise RuntimeError("; ".join(problems))
    except Exception as exc:
        raise RuntimeError(
            f"The graph is not ready for retrieval: {exc}\n"
            "Run Lab 1 (01-graph-build/1.1_build_graph.ipynb) first."
        ) from exc

    print(f"{CHUNK_VECTOR_INDEX} is ONLINE")
    print(f"{CHUNK_FULLTEXT_INDEX} is ONLINE")
    print("Every graph fact these questions depend on is present")
else:
    print("\nThe cells below will skip. Finish Lab 0 and Lab 1, then come back.")

## The schema the graph actually holds

Lab 1 pinned this schema during extraction, so the traversals below can name
relationships instead of discovering them. Render it before using any retriever
that traverses, so the shape of the added context is predictable.

In [ ]:
from workshop.graph_schema import GRAPH_SCHEMA

pattern_rows = "".join(
    f"<tr><td><strong>{source}</strong></td>"
    f"<td><code>-[:{relationship}]-&gt;</code></td>"
    f"<td><strong>{target}</strong></td></tr>"
    for source, relationship, target in GRAPH_SCHEMA["patterns"]
)
display(HTML(
    "<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>"
    f"</thead><tbody>{pattern_rows}</tbody></table>"
    "<p>Each extracted entity also points to its source "
    "<code>(entity)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>"
))

## One embedder, shared with the build

A query embedding has to come from the same model, at the same dimension, with
the same purpose as the vectors Lab 1 wrote. A mismatch does not raise. It
returns confident, wrong neighbours. So the embedder is constructed from the
same module the build used rather than configured again here.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from neo4j_graphrag.types import RetrieverResultItem

    from workshop.bedrock_providers import BedrockEmbeddings

    embedder = BedrockEmbeddings(region_name=os.environ.get("AWS_REGION", "us-east-1"))
    print(f"model: {EMBEDDING_MODEL_ID}")
    print(f"dimensions: {EMBEDDING_DIMENSIONS}")

    def chunk_text_formatter(record):
        """Return the chunk text itself.

        The library default is `content=str(node)`, which prints the whole node
        map with its newlines escaped and is unreadable on a projector.
        """
        node = record.get("node") or {}
        return RetrieverResultItem(
            content=node.get("text") or "",
            metadata={"score": record.get("score")},
        )

    def show_results(question, result, why):
        """Print each retrieved item with its score, then why the pattern fits."""
        print(f"Question: {question}\n")
        for number, item in enumerate(result.items, 1):
            score = (item.metadata or {}).get("score")
            score_text = "n/a" if score is None else f"{score:.4f}"
            content = str(item.content)
            preview = content[:700] + ("…" if len(content) > 700 else "")
            print(f"[{number}] score={score_text}\n{preview}\n")
        print(f"Why this fits: {why}")

## Pattern 1: plain vector retrieval

One question runs through both patterns below, at the same `top_k`, over the
same index, with the same embedding. The only thing that differs between the two
outputs is what happens after the vector search returns.

The question is the one the rest of the workshop keeps coming back to:

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

Plain vector retrieval is the right reach when the question is a paraphrase and
the answer sits in the text of a single chunk. The question paraphrases the
document rather than quoting it, which is what semantic similarity is for. What
comes back is chunk text: the rating and the amenity list are somewhere in that
prose, unlabelled, for a model to re-read and re-extract.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from neo4j_graphrag.retrievers import VectorRetriever

    from workshop.graph_setup import HERO_NAME

    HERO_QUESTION = f"What amenities and guest rating does {HERO_NAME} have?"
    TOP_K = 3

    vector_retriever = VectorRetriever(
        driver=driver,
        index_name=CHUNK_VECTOR_INDEX,
        embedder=embedder,
        return_properties=["text"],
        result_formatter=chunk_text_formatter,
    )
    vector_result = vector_retriever.search(query_text=HERO_QUESTION, top_k=TOP_K)
    show_results(
        HERO_QUESTION,
        vector_result,
        "Semantic similarity finds the document from a paraphrase of its wording, "
        "and returns its text and nothing else.",
    )

## Pattern 2: the same search, plus what the chunk is connected to

Same question, same `top_k`, same index, same embedding. `VectorCypherRetriever`
runs the identical vector search and then hands each match to a Cypher query you
wrote. The traversal below walks from the chunk to the hotel it came from, then
out to that hotel's rooms, amenities, policies, and services, keeping up to
three names per relationship type so no type gets sliced away.

The Cypher is static and reviewed. No part of it is generated by a model, and
nothing from the question is interpolated into it. That matters more than it
looks: this is the shape you can put behind a tool boundary and still reason
about what it can touch.

Two details in the code below are worth reading rather than skimming. The match
from chunk to hotel is `OPTIONAL`, so a chunk with no extracted hotel still
comes back as a result instead of silently disappearing from the list. And the
map projection asks for `hotel_id`, the opaque identifier Lab 1 stamped on the
fixture hotels, which is the identity Lab 4 writes against.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from neo4j_graphrag.retrievers import VectorCypherRetriever

    retrieval_query = """
    OPTIONAL MATCH (node)<-[:FROM_CHUNK]-(candidate:Hotel)
    WITH node, score, head(collect(candidate)) AS hotel
    CALL (hotel) {
        OPTIONAL MATCH (hotel)-[relationship]->(detail)
        WHERE type(relationship) IN [
            'HAS_ROOM', 'OFFERS_AMENITY', 'HAS_POLICY', 'PROVIDES_SERVICE'
        ]
          AND coalesce(detail.name, detail.type) IS NOT NULL
        WITH type(relationship) AS relationship_type,
             coalesce(detail.name, detail.type) AS name
        ORDER BY relationship_type, name
        WHERE relationship_type IS NOT NULL
        WITH relationship_type, collect(DISTINCT name)[..3] AS names
        ORDER BY relationship_type
        RETURN collect({relationship: relationship_type, names: names}) AS related
    }
    RETURN node.text AS chunk, score,
           hotel { .hotel_id, .name, .address, .guest_rating } AS hotel,
           related
    ORDER BY score DESC
    """

    def graph_result_formatter(record):
        """Keep the chunk as content and the graph fields as named metadata."""
        return RetrieverResultItem(
            content=record.get("chunk") or "",
            metadata={
                "score": record.get("score"),
                "hotel": record.get("hotel"),
                "related": record.get("related"),
            },
        )

    def show_graph_results(question, result, why):
        """Print the named graph fields first, then a short chunk preview."""
        print(f"Question: {question}\n")
        for number, item in enumerate(result.items, 1):
            metadata = item.metadata or {}
            score = metadata.get("score")
            score_text = "n/a" if score is None else f"{score:.4f}"
            hotel = metadata.get("hotel") or {}
            print(f"[{number}] score={score_text}")
            print(f"    hotel_id:     {hotel.get('hotel_id') or 'unset'}")
            print(f"    name:         {hotel.get('name') or 'no hotel on this chunk'}")
            print(f"    address:      {hotel.get('address') or 'unknown'}")
            print(f"    guest_rating: {hotel.get('guest_rating')}")
            for group in metadata.get("related") or []:
                names = ", ".join(group.get("names") or [])
                print(f"    {group.get('relationship')}: {names}")
            chunk = str(item.content)
            preview = chunk[:300] + ("…" if len(chunk) > 300 else "")
            print(f"    chunk: {preview}\n")
        print(f"Why this fits: {why}")

    def scores_of(result):
        """Pull the raw score list, which is how the two patterns are compared."""
        return [(item.metadata or {}).get("score") for item in result.items]

    def score_line(result):
        """Render one result set's scores for printing."""
        return ", ".join(
            "n/a" if score is None else f"{score:.4f}" for score in scores_of(result)
        )

    vector_cypher_retriever = VectorCypherRetriever(
        driver=driver,
        index_name=CHUNK_VECTOR_INDEX,
        retrieval_query=retrieval_query,
        embedder=embedder,
        result_formatter=graph_result_formatter,
    )
    graph_result = vector_cypher_retriever.search(
        query_text=HERO_QUESTION,
        top_k=TOP_K,
    )
    show_graph_results(
        HERO_QUESTION,
        graph_result,
        "Vector search locates the same chunks; Cypher adds the connected entities "
        "and typed relationships as named fields.",
    )
    print(f"\nPattern 1 scores: {score_line(vector_result)}")
    print(f"Pattern 2 scores: {score_line(graph_result)}")

    # The claim below is checked rather than printed on faith: if the two
    # patterns ever stop running the same vector search, this fails loudly.
    vector_scores = scores_of(vector_result)
    graph_scores = scores_of(graph_result)
    assert len(vector_scores) == len(graph_scores) and all(
        first is not None and second is not None and abs(first - second) < 1e-6
        for first, second in zip(vector_scores, graph_scores)
    ), "the two patterns did not return the same scores in the same order"
    print("Same scores, in the same order: it is the same vector search both times.")

## What changed

Both retrievers found the same chunks, because they ran the same vector search:
one question, one embedding, one index, one `top_k`, and the printed scores line
up. Only the second one came back with the hotel's identifier, its rating, its
amenity names, and a few names from each of its other relationship types as
named fields, rather than as prose an answering model would have to re-read and
re-extract.

That is the whole delta, and it decides two things later in the workshop. An
agent that gets `guest_rating: 4.5` as a field cannot round it to 4 or invent a
5. And a write path that gets `hotel_id` back can act on the right hotel without
matching on a display name. Only the fixture hotels carry a `hotel_id` today,
because Lab 1 stamps it deliberately rather than letting extraction invent one,
so any other chunk in the list prints `unset`.

| Ask this | Reach for | Because |
|---|---|---|
| A paraphrased fact stated in one chunk | `VectorRetriever` | Meaning matters more than wording, and there is nothing to traverse |
| The same, plus the entity's connected facts | `VectorCypherRetriever` | One vector hop finds the chunk, one reviewed traversal finds the rest |

**Next:** `2.2_fulltext_retrievers.ipynb` takes on the question vector search is
worst at, an exact identifier, and builds the retriever Lab 5 deploys unchanged.

In [ ]:
if RETRIEVAL_READY:
    driver.close()
    print("This notebook's driver is closed.")